In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import warnings

warnings.filterwarnings("ignore", category=RuntimeWarning)

mpl.rcParams.update(
    {
        "text.usetex": False,
        "axes.labelsize": 20,
        "figure.labelsize": 18,
        "xtick.labelsize": 16,
        "ytick.labelsize": 16,
        "figure.constrained_layout.wspace": 0,
        "figure.constrained_layout.hspace": 0,
        "figure.constrained_layout.h_pad": 0,
        "figure.constrained_layout.w_pad": 0,
        "axes.linewidth": 1.2,
    }
)

import jax
import jax.numpy as jnp

# you should always set this
jax.config.update("jax_enable_x64", True)

### Multiband Fitting and Prediction: Filling the Gaps in Each Band
This notebook shows how joint multiband fitting with **EzTaoX** can recover the light curve in a band over a time range where that band has **no observations**, as long as another band is observed during that range. Ground-based surveys produce this situation all the time: a filter is unavailable for part of a season, a detector fails, or the cadence simply differs between bands.

The idea is simple. In the **EzTaoX** multiband model, all bands are generated by the same latent Gaussian process (GP), with band-dependent amplitudes and lags (see the [Multiband Fitting](03_MultibandFitting.ipynb) tutorial). The covariance between an observation at time $t_i$ in band $a$ and one at time $t_j$ in band $b$ is

$$
k_{ab}(t_i, t_j) = S_a S_b \, k_{\rm latent}\left(|t_j - t_i - (\tau_b - \tau_a)|\right),
$$

where $k_{\rm latent}$ is the latent kernel (a damped random walk, DRW, in this notebook, with timescale $\tau_{\rm DRW}$ and amplitude $\sigma_{\rm DRW}$), $S_a$ is the amplitude scale of band $a$, and $\tau_a$ is its lag relative to the reference band. Once the model is fitted, the conditional GP prediction in one band is informed by the observations in *every* band, so a band's gap can be filled using the other band's observations taken during that gap.

<div class="alert alert-info">

**Note**

Band 0 is the reference band: its amplitude scale is fixed to $S_0 = 1$ and its lag to $\tau_0 = 0$, so the model only fits $S$ and $\tau$ for the remaining $M-1$ bands. Positive parameters are stored in log space in the parameter dictionary.

</div>

We will:

1. simulate a two-band (`g`, `r`) DRW light curve,
2. remove the observations in a different 400-day window in each band, so that the two gaps do not overlap,
3. fit both bands jointly, and
4. predict both bands across their gaps and compare the predictions to the (noise-free) truth.

In [ ]:
from eztaox.kernels.quasisep import Exp
from eztaox.simulator import MultiVarSim
from eztaox.ts_utils import add_noise, formatlc
from eztaox.models import MultiVarModel
from eztaox.fitter import random_search

import numpyro
import numpyro.distributions as dist

#### 1. Light Curve Simulation
We use `MultiVarSim` (see the [Simulating Multi-band Light Curves](04_MultibandSimulation.ipynb) tutorial) with a DRW latent kernel. The two bands share the DRW timescale, while the `r` band has a smaller amplitude and lags the `g` band by a few days.

Two realizations are drawn from the **same** random key: the `full` light curve on the simulator grid (one point per day), which we keep as the noise-free truth, and a `random` subsample of it, which plays the role of the observations.

In [ ]:
band_order = {"g": 0, "r": 1}
band_names = tuple(band_order.keys())
band_colors = {"g": "tab:green", "r": "tab:red"}

# latent DRW parameters
drw_scale = 150.0
drw_sigma = 0.2

# observed-band amplitudes and lags (band g is the reference band)
amp_scale = {"g": 1.0, "r": 0.8}
lag = {"g": 0.0, "r": 5.0}

sim_params = {
    "log_kernel_param": jnp.log(jnp.asarray([drw_scale, drw_sigma])),
    "log_amp_scale": jnp.log(jnp.asarray([amp_scale["r"]])),
    "lag": jnp.asarray([lag["r"]]),
}

min_dt, max_dt = 1.0, 2000.0
kernel = Exp(scale=drw_scale, sigma=drw_sigma)

simulator = MultiVarSim(
    kernel,
    min_dt,
    max_dt,
    n_band=len(band_order),
    init_params=sim_params,
    zero_mean=True,
    has_lag=True,
)

In [ ]:
def split_by_band(X, y, band_order):
    """Split (X, y) in EzTaoX format into per-band, time-sorted dictionaries."""
    ts, ys = {}, {}

    for band, idx in band_order.items():
        mask = X[1] == idx
        order = jnp.argsort(X[0][mask])
        ts[band] = X[0][mask][order]
        ys[band] = y[mask][order]

    return ts, ys

In [ ]:
lc_key = jax.random.PRNGKey(7)
n_obs = 1000  # total number of observations across both bands

# noise-free truth on the daily grid
truth_X, truth_y = simulator.full(lc_key)
truth_ts, truth_ys = split_by_band(truth_X, truth_y, band_order)

# observations: random subsample of the same realization, plus photometric noise
obs_X, obs_y = simulator.random(n_obs, lc_key, jax.random.PRNGKey(1))
obs_ts, obs_ys = split_by_band(obs_X, obs_y, band_order)

noise_level = {"g": 0.03, "r": 0.03}
ts, ys, yerrs = {}, {}, {}

for i, band in enumerate(band_names):
    ts[band] = obs_ts[band]
    yerrs[band] = jnp.ones_like(obs_ts[band]) * noise_level[band]
    ys[band] = add_noise(obs_ys[band], yerrs[band], jax.random.PRNGKey(10 + i))

{band: ts[band].size for band in band_names}

##### 1.1 Remove the observations in a different window in each band
We now drop every `g`-band point between day 500 and day 900, and every `r`-band point between day 1200 and day 1600. Each gap is roughly 2.7 DRW timescales long, so the data on either side of a gap carry very little information about what happens inside it. The two gaps do not overlap: whenever one band is missing, the other band is still observed.

In [ ]:
gaps = {"g": (500.0, 900.0), "r": (1200.0, 1600.0)}

for band in band_names:
    keep = (ts[band] < gaps[band][0]) | (ts[band] > gaps[band][1])
    print(f"{band}-band points: {ts[band].size} -> {int(keep.sum())}")
    ts[band], ys[band], yerrs[band] = ts[band][keep], ys[band][keep], yerrs[band][keep]

To show both bands on the same axes, we plot them with a constant vertical offset. The shaded windows mark each band's gap, in the band's color. The gray line is the noise-free truth, which we can see inside the gaps even though no data were taken there. The same helper function also draws the GP prediction later on.

In [ ]:
offset = {"g": -0.5, "r": 0.5}


def plot_bands(means=None, stds=None):
    """Plot the data and truth in each band with an offset, plus an optional GP prediction."""
    # data
    for band in band_names:
        plt.errorbar(
            ts[band],
            ys[band] + offset[band],
            yerrs[band],
            fmt=".",
            c=band_colors[band],
            label=f"{band}-band",
            zorder=10,
        )

    for band in band_names:
        # the band's gap
        plt.axvspan(*gaps[band], color=band_colors[band], alpha=0.1, zorder=0)

        # truth
        plt.plot(
            truth_ts[band],
            truth_ys[band] + offset[band],
            c="gray",
            lw=1,
            alpha=0.6,
            label="Truth" if band == band_names[0] else None,
            zorder=4 if means is not None else 1,
        )

        # GP prediction
        if means is None:
            continue

        plt.plot(
            truth_ts[band],
            means[band] + offset[band],
            c="tab:blue",
            lw=2,
            label="GP prediction" if band == band_names[0] else None,
            zorder=3,
        )
        plt.fill_between(
            truth_ts[band],
            means[band] - stds[band] + offset[band],
            means[band] + stds[band] + offset[band],
            color="tab:blue",
            alpha=0.3,
            zorder=1,
        )

    plt.gca().invert_yaxis()
    plt.xlim(0, max_dt)
    plt.xlabel("Time (day)")
    plt.ylabel("Flux (mag) + offset")
    plt.legend(
        fontsize=15,
        ncol=4,
        loc="lower center",
        bbox_to_anchor=(0.5, 1.0),
        frameon=False,
    )


plt.figure(figsize=(9, 6))
plot_bands()

#### 2. Light Curve Formatting
As in the [Multiband Fitting](03_MultibandFitting.ipynb) tutorial, `formatlc` turns the per-band dictionaries into the `(X, y, yerr)` triple expected by `MultiVarModel`, where `X = (time, band index)`.

In [ ]:
X, y, yerr = formatlc(ts, ys, yerrs, band_order)
X[0].shape, y.shape

#### 3. Light Curve Fitting
We fit both bands jointly with `MultiVarModel`, using `random_search` for the maximum-likelihood fit. `random_search` needs an `init_sampler` that draws random starting points in the parameter space.

**Model parameters:**
- **log_kernel_param**: The parameters of the latent GP process, in natural log. For the DRW kernel these are $\log \tau_{\rm DRW}$ and $\log \sigma_{\rm DRW}$.
- **log_amp_scale**: The log of the amplitude ratio between each band and the latent GP (the $S$ factor). Band 0 has $S = 1$, so this array has size `M-1 = 1`.
- **lag**: The lag of each band relative to the reference band, also of size `M-1 = 1`.
- **mean**: The per-band light curve mean, of size `M`. It is omitted here because the model is built with `zero_mean=True`.

In [ ]:
def init_sampler():
    # GP kernel param
    log_drw_scale = numpyro.sample(
        "log_drw_scale", dist.Uniform(jnp.log(1.0), jnp.log(2000.0))
    )
    log_drw_sigma = numpyro.sample(
        "log_drw_sigma", dist.Uniform(jnp.log(0.01), jnp.log(2.0))
    )
    log_kernel_param = jnp.stack([log_drw_scale, log_drw_sigma])
    numpyro.deterministic("log_kernel_param", log_kernel_param)

    # amplitude ratio and lag of the r band relative to the g band
    log_amp_scale = numpyro.sample("log_amp_scale", dist.Uniform(-2.0, 2.0))
    lag = numpyro.sample("lag", dist.Uniform(-20.0, 20.0))

    return {
        "log_kernel_param": log_kernel_param,
        "log_amp_scale": log_amp_scale,
        "lag": lag,
    }

In [ ]:
%%time
fit_key = jax.random.PRNGKey(2)
n_sample = 1_000
n_best = 5

k = Exp(scale=100.0, sigma=1.0)  # init params for k are not used
joint_model = MultiVarModel(
    X, y, yerr, k, n_band=len(band_order), has_lag=True, zero_mean=True
)

joint_bestP, joint_ll = random_search(
    joint_model, init_sampler, fit_key, n_sample, n_best
)
print("True DRW (scale, sigma):", drw_scale, drw_sigma)
print("Best-fit DRW (scale, sigma):", jnp.exp(joint_bestP["log_kernel_param"]))
print(
    "True / best-fit r-band amplitude scale:",
    amp_scale["r"],
    jnp.exp(joint_bestP["log_amp_scale"]),
)
print("True / best-fit r-band lag:", lag["r"], joint_bestP["lag"])

#### 4. Light Curve Prediction
The `pred` method returns the conditional GP mean and standard deviation at the requested inputs. For `MultiVarModel`, `pred` takes `X` in the same `(time, band index)` format as the training data, so you choose which band to predict by setting the band index. The lags are applied automatically, so the predicted times refer to the observer frame of the requested band. Here we predict both bands on the daily grid in a single call and split the result by band.

In [ ]:
n_grid = truth_ts["g"].size
pred_X = (
    jnp.concatenate([truth_ts[band] for band in band_names]),
    jnp.concatenate(
        [jnp.full(n_grid, band_order[band], dtype=int) for band in band_names]
    ),
)
all_mean, all_std = joint_model.pred(joint_bestP, pred_X)

joint_mean, joint_std = {}, {}
for band in band_names:
    sl = slice(band_order[band] * n_grid, (band_order[band] + 1) * n_grid)
    joint_mean[band], joint_std[band] = all_mean[sl], all_std[sl]

In [ ]:
plt.figure(figsize=(9, 6))
plot_bands(joint_mean, joint_std)

The gray line is the noise-free truth, and the blue line and shaded region are the GP prediction and its $\pm 1\sigma$ uncertainty. Both gaps are filled. Inside the `g`-band gap, the `r`-band observations pin down the latent GP, and vice versa, so the prediction in each band tracks the truth across its gap.

##### 4.1 Zoom in on the gaps
Let us zoom in on each gap to compare the GP prediction with the truth in more detail. The top panel shows the `g`-band gap and the bottom panel the `r`-band gap; the two panels cover different time ranges.

In [ ]:
pad = 150.0

fig, axes = plt.subplots(2, 1, figsize=(9, 9), layout="constrained")

for ax, band in zip(axes, band_names):
    lo, hi = gaps[band][0] - pad, gaps[band][1] + pad
    t = truth_ts[band]
    zoom = (t >= lo) & (t <= hi)
    in_window = (ts[band] >= lo) & (ts[band] <= hi)

    # data, truth, and GP prediction
    ax.errorbar(
        ts[band][in_window],
        ys[band][in_window],
        yerrs[band][in_window],
        fmt=".",
        c=band_colors[band],
        label=f"{band}-band",
        zorder=10,
    )
    ax.plot(
        t[zoom],
        truth_ys[band][zoom],
        c="gray",
        lw=1,
        alpha=0.6,
        label="Truth" if band == band_names[0] else None,
        zorder=4,
    )
    ax.plot(
        t[zoom],
        joint_mean[band][zoom],
        c="tab:blue",
        lw=2,
        label="GP prediction" if band == band_names[0] else None,
        zorder=3,
    )
    ax.fill_between(
        t[zoom],
        (joint_mean[band] - joint_std[band])[zoom],
        (joint_mean[band] + joint_std[band])[zoom],
        color="tab:blue",
        alpha=0.3,
        zorder=1,
    )
    ax.axvspan(*gaps[band], color=band_colors[band], alpha=0.1, zorder=0)

    ax.invert_yaxis()
    ax.set_xlim(lo, hi)
    ax.set_ylabel("Flux (mag)")

axes[-1].set_xlabel("Time (day)")

# one legend above the top panel, collecting the entries from both panels
handles, labels = [
    sum(x, []) for x in zip(*(ax.get_legend_handles_labels() for ax in axes))
]
axes[0].legend(
    handles,
    labels,
    fontsize=15,
    ncol=4,
    loc="lower center",
    bbox_to_anchor=(0.5, 1.0),
    frameon=False,
)

Inside each gap, the prediction follows the variability of the truth and the $\pm 1\sigma$ band stays narrow, because the other band was observed throughout. This is the practical payoff of the shared latent GP: every band contributes to the prediction in every other band, so coverage in one band fills the gaps in another.